# ⚖️ Legal Tech Agentic AI

## Intelligent Multi-Agent Legal Assistant

This notebook demonstrates an Agentic AI system capable of

- Contract Review
- Legal Research
- Compliance Checking
- Case Law Analysis
- Legal Risk Assessment
- Report Generation

### Architecture

User
 ↓
Coordinator Agent
 ├── Contract Review Agent
 ├── Legal Research Agent
 ├── Compliance Agent
 ├── Case Law Agent
 ├── Risk Assessment Agent
 └── Report Generator

In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-chroma
!pip install sentence-transformers
!pip install chromadb
!pip install pymupdf
!pip install pypdf
!pip install pandas
!pip install ollama
!pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 89.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 101.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 92.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 84.7 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 104.5 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 106.2 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 107.3 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 103.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 94.0 MB/s  0:00:02m0:0

In [2]:
!pip install -U langchain-huggingface

In [3]:
import os
import pandas as pd
import fitz

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader

from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.llms import Ollama

/tmp/ipykernel_108/528815487.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded successfully


In [5]:
pdf_folder = "legal_docs"

documents = []

for filename in os.listdir(pdf_folder):

    if filename.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join(pdf_folder, filename))

        docs = loader.load()

        documents.extend(docs)

print(f"✅ Loaded {len(documents)} pages")

✅ Loaded 623 pages


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Created {len(chunks)} chunks")

✅ Created 2211 chunks


In [7]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="legal_db"
)

print("✅ Chroma Database Created")

✅ Chroma Database Created


In [8]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":5}
)

print("✅ Retriever Ready")

✅ Retriever Ready


In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0
)

print("LLM Loaded Successfully")

LLM Loaded Successfully


In [10]:
query = "What are the obligations of the employer?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\n========== Document {i} ==========\n")
    print(doc.page_content[:600])


========== Document 1 ==========

may be necessary to enable them to function as units of self-government. 
41. Right to work, to educ ation and to public assistance in certain 
cases.—The State shall, within the limits of its economic capacity and 
development, make effective provision for securing the right to work, to 
education and to public assistance in cases of unemployment, old age, sickness 
and disablement, and in other cases of undeserved want. 
42. Provision for just and humane conditions of work  and maternity 
relief.—The State shall make provision for securing just and humane conditions 
of work and for maternit

========== Document 2 ==========

may be necessary to enable them to function as units of self-government. 
41. Right to work, to educ ation and to public assistance in certain 
cases.—The State shall, within the limits of its economic capacity and 
development, make effective provision for securing the right to work, to 
education and to public assistance in c

In [11]:
def contract_review_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are an experienced Contract Review Lawyer.

Review the contract and provide:

1. Summary
2. Key Obligations
3. Important Clauses
4. Missing Clauses
5. Potential Legal Issues

Contract:

{context}
"""

    return llm.invoke(prompt)

In [12]:
response = contract_review_agent(
    "Review this employment agreement."
)

print(response.content)

**Review of the Indian Contract Act, 1872 (Sections 1–259)**  
*Note: The text provided is the full text of the Indian Contract Act, 1872, not a specific contract between parties. However, I will analyze the Act’s provisions as if they were part of a hypothetical contract, focusing on its legal framework and implications.*

---

### **1. Summary**  
The **Indian Contract Act, 1872** is a foundational legal document governing contracts in India. It outlines the **legal requirements for valid contracts**, **voidable agreements**, **partnership rules**, and **enforceability of obligations**. Key sections define:  
- **Communication, acceptance, and revocation of proposals** (Chapters I and II).  
- **Competency of parties** (e.g., age, mental capacity).  
- **Voidable contracts** due to coercion, undue influence, fraud, or misrepresentation.  
- **Partnership obligations** (Sections 252–259), including dissolution, duties, and rights of partners.  

The Act serves as a **legal framework**

In [13]:
def legal_research_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are an experienced Legal Research Assistant.

Use ONLY the provided legal documents.

Your job is to answer:

1. Applicable Law
2. Relevant Sections
3. Legal Interpretation
4. Practical Explanation
5. Conclusion

If the answer is unavailable in the documents,
say:

"Not found in the provided legal documents."

Question:

{query}

Legal Documents:

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [14]:
print(
    legal_research_agent(
        "Explain Section 10 of the Indian Contract Act."
    )
)

1. **Applicable Law**:  
   The question pertains to **Section 10 of the Indian Contract Act, 1872**, which is a foundational provision governing the formation of valid contracts.  

2. **Relevant Sections**:  
   The provided legal documents do **not include Section 10** of the Indian Contract Act. The text includes sections from the **Constitution of India** (e.g., Articles 299 and 300) and the **preliminary sections** of the Indian Contract Act (e.g., Sections 1 and 2). However, **Section 10** is absent from the provided content.  

3. **Legal Interpretation**:  
   Since Section 10 is not present in the provided documents, no interpretation can be derived from it.  

4. **Practical Explanation**:  
   Section 10 of the Indian Contract Act typically defines the **essence of a valid contract**, stating that an agreement is enforceable as a contract if:  
   - It is made between two or more parties.  
   - It is lawful in itself.  
   - It is made with a **lawful consideration** (some

In [15]:
def compliance_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a Legal Compliance Officer.

Analyse the document for legal compliance.

Return your answer in this format.

Compliance Status

Applicable Regulations

Violations

Missing Requirements

Recommendations

Compliance Score (0-100)

Document:

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [16]:
print(
    compliance_agent(
        "Check GDPR compliance."
    )
)

Compliance Status  
Compliant  

Applicable Regulations  
- General Data Protection Regulation (GDPR) (EU) 2016/679  

Violations  
None identified. The text is a direct excerpt from the GDPR, which is a binding regulation.  

Missing Requirements  
None. The document is the GDPR itself, which is a comprehensive legal framework.  

Recommendations  
Ensure that the document is implemented correctly in organizational policies and practices to align with GDPR requirements.  

Compliance Score (0-100)  
100


In [17]:
def case_law_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a Senior Legal Case Analyst.

Based on the retrieved documents provide

Relevant Legal Principles

Applicable Sections

Possible Similar Cases

Legal Interpretation

Possible Outcome

Question

{query}

Documents

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [18]:
print(
    case_law_agent(
        "What happens if a contract is signed under coercion?"
    )
)

### **Relevant Legal Principles**  
1. **Coercion under the Indian Contract Act (Section 15):**  
   A contract is void if it is entered into under **coercion**, which includes threats, fear of injury, or other forms of pressure that compel a party to agree against their will.  

2. **Undue Influence (Section 23):**  
   A contract is voidable at the option of the aggrieved party if it is induced by **undue influence**, where one party is in a position to dominate the will of the other and uses that position to gain an unfair advantage.  

3. **Criminal Intimidation under the Indian Penal Code (IPC, Section 506):**  
   Coercion may also constitute an offense under the IPC, even if the IPC is not in force in the jurisdiction where the act occurred (as per the illustration in the documents).  

---

### **Applicable Sections**  
1. **Indian Contract Act, 1872:**  
   - **Section 15:** Contracts made under coercion are void.  
   - **Section 23:** Contracts induced by undue influence are

In [19]:
def risk_assessment_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a Legal Risk Assessment Expert.

Identify

High Risks

Medium Risks

Low Risks

Potential Liabilities

Risk Score (1-10)

Recommendations

Document

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [20]:
print(
    risk_assessment_agent(
        "Assess risks in this employment agreement."
    )
)

### **Legal Risk Assessment Report**  
**Document:** Legal Provisions on District Commission Powers and Suretyship Obligations  

---

### **1. High Risks (Risk Score: 8–10)**  
**a. Product Liability and Punitive Damages**  
- **Risk:** The District Commission’s authority to grant **punitive damages** and **compensation** under Chapter VI could result in **severe financial penalties** if a company fails to address product defects, unsafe goods, or unfair trade practices.  
- **Potential Liability:** Legal action, reputational damage, and operational shutdowns (e.g., ceasing manufacture of hazardous goods).  
- **Key Provisions:**  
  - **(e)** Compensation for product liability claims.  
  - **(h)-(j)** Mandatory cessation of hazardous goods, unsafe practices, or unfair trade practices.  
  - **(k)** Minimum 25% payment for large-scale consumer harm.  

**b. Non-Compliance with Unfair Trade Practices**  
- **Risk:** Failure to **discontinue unfair/restrictive trade practices** could l

In [21]:
def coordinator_agent(query):

    query = query.lower()

    if any(word in query for word in [
        "contract",
        "agreement",
        "review",
        "clause",
        "employment"
    ]):

        return contract_review_agent(query)

    elif any(word in query for word in [
        "section",
        "law",
        "legal",
        "research",
        "act",
        "explain"
    ]):

        return legal_research_agent(query)

    elif any(word in query for word in [
        "compliance",
        "gdpr",
        "policy",
        "regulation",
        "iso"
    ]):

        return compliance_agent(query)

    elif any(word in query for word in [
        "case",
        "court",
        "judgement",
        "precedent"
    ]):

        return case_law_agent(query)

    elif any(word in query for word in [
        "risk",
        "liability",
        "danger",
        "penalty"
    ]):

        return risk_assessment_agent(query)

    else:

        return legal_research_agent(query)

In [22]:
query = input("Enter your legal question: ")

answer = coordinator_agent(query)

print("\n")
print("="*80)
print(answer)
print("="*80)

Enter your legal question:  Check GDPR compliance for this document.




Compliance Status  
**Non-Compliant**  

Applicable Regulations  
- **GDPR (Regulation (EU) 2016/679)**  
- **Directive 95/46/EC** (now obsolete, replaced by GDPR)  
- **Regulation (EC) No 45/2001** (European Data Protection Supervisor)  

Violations  
1. **Missing Contact Details**: The document fails to provide the required contact details of the data protection office (e.g., "data prot ection office r" is incomplete).  
2. **Outdated References**: References to Directive 95/46/EC (repealed by GDPR) are present, which may conflict with current GDPR requirements.  
3. **OCR Errors**: Text contains errors (e.g., "necessar y", "ag ain", "Official Jour nal") that compromise readability and accuracy.  
4. **Incomplete Legal Context**: The document lacks clarity on how transitional provisions (e.g., Article 28(2) of Regulation (EC) No 45/2001) apply to GDPR compliance.  

Missing Requirements  
1. **Contact Details**: Specific contact information for the data protection office (e.g., nam

In [27]:
from typing import TypedDict

from langgraph.graph import StateGraph, END

In [29]:
class LegalState(TypedDict):
    query: str
    contract_review: str
    legal_research: str
    compliance: str
    risk: str
    final_report: str

In [30]:
def planner_node(state):

    print("Planning Legal Analysis...")

    return state

In [31]:
def contract_node(state):

    result = contract_review_agent(state["query"])

    state["contract_review"] = result

    return state

In [32]:
def research_node(state):

    result = legal_research_agent(state["query"])

    state["legal_research"] = result

    return state

In [33]:
def compliance_node(state):

    result = compliance_agent(state["query"])

    state["compliance"] = result

    return state

In [34]:
def risk_node(state):

    result = risk_assessment_agent(state["query"])

    state["risk"] = result

    return state

In [35]:
def final_report_node(state):

    prompt = f"""
You are a Senior Legal Advisor.

Combine the following analyses into one professional report.

Contract Review
----------------
{state["contract_review"]}

Legal Research
----------------
{state["legal_research"]}

Compliance
----------------
{state["compliance"]}

Risk Assessment
----------------
{state["risk"]}

Generate a professional report containing:

1. Executive Summary

2. Contract Analysis

3. Legal Research Findings

4. Compliance Review

5. Risk Analysis

6. Final Recommendations

7. Overall Legal Opinion
"""

    response = llm.invoke(prompt)

    state["final_report"] = response.content

    return state

In [36]:
workflow = StateGraph(LegalState)

workflow.add_node("Planner", planner_node)
workflow.add_node("Contract", contract_node)
workflow.add_node("Research", research_node)
workflow.add_node("Compliance", compliance_node)
workflow.add_node("Risk", risk_node)
workflow.add_node("Report", final_report_node)

workflow.set_entry_point("Planner")

workflow.add_edge("Planner", "Contract")
workflow.add_edge("Contract", "Research")
workflow.add_edge("Research", "Compliance")
workflow.add_edge("Compliance", "Risk")
workflow.add_edge("Risk", "Report")
workflow.add_edge("Report", END)

In [37]:
legal_graph = workflow.compile()

print("LangGraph Workflow Ready!")

LangGraph Workflow Ready!


In [ ]:
query = input("Enter your legal query: ")

result = legal_graph.invoke(
    {
        "query": query,
        "contract_review": "",
        "legal_research": "",
        "compliance": "",
        "risk": "",
        "final_report": ""
    }
)

print("\n")
print("=" * 80)
print(result["final_report"])
print("=" * 80)